In [1]:
1+1

2

In [2]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('dataset.csv')

# Print the columns and the first few rows
print("Columns in dataset:", df.columns.tolist())
print("\nFirst 3 rows:")
display(df.head(3))

# Check how many missing values we have in our target column
print("\nMissing IV values:", df['Implied Volatility'].isna().sum())

Columns in dataset: ['datetime', 'underlying_price', 'NIFTY27JAN2625200CE', 'NIFTY27JAN2625300CE', 'NIFTY27JAN2625400CE', 'NIFTY27JAN2625500CE', 'NIFTY27JAN2625600CE', 'NIFTY27JAN2625700CE', 'NIFTY27JAN2625800CE', 'NIFTY27JAN2625900CE', 'NIFTY27JAN2626000CE', 'NIFTY27JAN2626100CE', 'NIFTY27JAN2626200CE', 'NIFTY27JAN2626300CE', 'NIFTY27JAN2626400CE', 'NIFTY27JAN2626500CE', 'NIFTY27JAN2623800PE', 'NIFTY27JAN2623900PE', 'NIFTY27JAN2624000PE', 'NIFTY27JAN2624100PE', 'NIFTY27JAN2624200PE', 'NIFTY27JAN2624300PE', 'NIFTY27JAN2624400PE', 'NIFTY27JAN2624500PE', 'NIFTY27JAN2624600PE', 'NIFTY27JAN2624700PE', 'NIFTY27JAN2624800PE', 'NIFTY27JAN2624900PE', 'NIFTY27JAN2625000PE', 'NIFTY27JAN2625100PE']

First 3 rows:


,datetime,underlying_price,NIFTY27JAN2625200CE,NIFTY27JAN2625300CE,NIFTY27JAN2625400CE,NIFTY27JAN2625500CE,NIFTY27JAN2625600CE,NIFTY27JAN2625700CE,NIFTY27JAN2625800CE,NIFTY27JAN2625900CE,...,NIFTY27JAN2624200PE,NIFTY27JAN2624300PE,NIFTY27JAN2624400PE,NIFTY27JAN2624500PE,NIFTY27JAN2624600PE,NIFTY27JAN2624700PE,NIFTY27JAN2624800PE,NIFTY27JAN2624900PE,NIFTY27JAN2625000PE,NIFTY27JAN2625100PE
0,07-01-2026 09:15,26111.65,0.12662,0.1233,0.11741,NaN,0.11005,0.10576,NaN,0.09724,...,0.15760,0.1524,0.14697,0.14105,0.13613,0.13085,0.12640,0.12142,0.11631,0.11150
1,07-01-2026 09:20,26141.40,0.08632,NaN,NaN,0.11779,0.11197,0.11028,NaN,NaN,...,NaN,0.1542,0.14753,0.14274,0.13849,0.13282,NaN,0.12363,NaN,0.11353
2,07-01-2026 09:25,26139.35,0.09147,NaN,0.09514,0.09933,0.09599,0.09204,0.09216,0.08954,...,0.15927,NaN,0.14919,0.14245,0.13806,0.13242,0.12877,0.12349,0.11817,NaN


KeyError: 'Implied Volatility'

In [3]:
import pandas as pd
import numpy as np

# 1. Load the dataset
df = pd.read_csv('dataset.csv')

# 2. Reshape the data from "Wide" to "Long" format
# We keep datetime and underlying_price, and melt the ticker columns
id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df.columns if col not in id_vars]

df_long = pd.melt(df, id_vars=id_vars, value_vars=ticker_cols, 
                  var_name='Ticker', value_name='Implied Volatility')

# 3. Extract the Strike price from the Ticker string using regex
# Tickers look like NIFTY27JAN2625200CE. We extract the digits right before 'CE' or 'PE'.
df_long['Strike'] = df_long['Ticker'].str.extract(r'(\d+)(?:CE|PE)$').astype(float)

# 4. Calculate Moneyness (Strike / Spot Price)
df_long['Moneyness'] = df_long['Strike'] / df_long['underlying_price']

# 5. Verify the transformation
print("Shape of reshaped data:", df_long.shape)
print("\nFirst 5 rows of our properly formatted data:")
display(df_long.head(5))
print("\nTotal Missing IV values to predict:", df_long['Implied Volatility'].isna().sum())

Shape of reshaped data: (27300, 6)

First 5 rows of our properly formatted data:


,datetime,underlying_price,Ticker,Implied Volatility,Strike,Moneyness
0,07-01-2026 09:15,26111.65,NIFTY27JAN2625200CE,0.12662,2625200.0,100.537500
1,07-01-2026 09:20,26141.40,NIFTY27JAN2625200CE,0.08632,2625200.0,100.423084
2,07-01-2026 09:25,26139.35,NIFTY27JAN2625200CE,0.09147,2625200.0,100.430959
3,07-01-2026 09:30,26128.95,NIFTY27JAN2625200CE,0.10860,2625200.0,100.470934
4,07-01-2026 09:35,26131.90,NIFTY27JAN2625200CE,0.10462,2625200.0,100.459592



Total Missing IV values to predict: 5460


In [4]:
from scipy.interpolate import RBFInterpolator
import warnings
warnings.filterwarnings('ignore')

# 1. FIX THE STRIKE AND MONEYNESS
# We extract exactly 5 digits for the strike (e.g., 25200) before CE/PE
df_long['Strike'] = df_long['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df_long['Moneyness'] = df_long['Strike'] / df_long['underlying_price']

print("Fixed Moneyness check (should be near 1.0):")
display(df_long[['Ticker', 'Strike', 'underlying_price', 'Moneyness']].head(3))

# 2. IMPLEMENT RBF INTERPOLATION
df_long['RBF_Pred'] = np.nan
timestamps = df_long['datetime'].unique()

print(f"\nStarting RBF Interpolation across {len(timestamps)} timestamps...")

for ts in timestamps:
    # Isolate data for this specific timestamp
    ts_data = df_long[df_long['datetime'] == ts]
    
    known = ts_data[ts_data['Implied Volatility'].notna()]
    missing = ts_data[ts_data['Implied Volatility'].isna()]
    
    # If there's nothing to predict, or not enough points to fit a curve, skip
    if len(missing) == 0 or len(known) < 3:
        continue
        
    # We fit the surface using our fixed Moneyness coordinate
    X_train = known[['Moneyness']].values
    y_train = known['Implied Volatility'].values
    X_predict = missing[['Moneyness']].values
    
    try:
        # Fit the RBF Interpolator with smoothing=0.0001 to prevent edge explosions
        rbf = RBFInterpolator(
            X_train, 
            y_train, 
            kernel='thin_plate_spline', 
            smoothing=0.0001 
        )
        
        # Predict the missing values
        preds = rbf(X_predict)
        
        # Clip predictions to prevent unrealistic financial values (e.g., negative IV)
        preds = np.clip(preds, a_min=0.001, a_max=3.0) 
        
        # Assign predictions back to the main dataframe
        df_long.loc[missing.index, 'RBF_Pred'] = preds
        
    except Exception as e:
        print(f"Failed to fit RBF for timestamp {ts}: {e}")

# 3. VERIFY RBF RESULTS
filled_count = df_long['RBF_Pred'].notna().sum()
print(f"\nSuccessfully generated {filled_count} RBF predictions!")

Fixed Moneyness check (should be near 1.0):


,Ticker,Strike,underlying_price,Moneyness
0,NIFTY27JAN2625200CE,25200.0,26111.65,0.965086
1,NIFTY27JAN2625200CE,25200.0,26141.40,0.963988
2,NIFTY27JAN2625200CE,25200.0,26139.35,0.964064



Starting RBF Interpolation across 975 timestamps...

Successfully generated 5460 RBF predictions!


In [5]:
# 1. Merge our predictions into a final IV column
df_long['Final_IV'] = df_long['Implied Volatility'].fillna(df_long['RBF_Pred'])

# 2. Pivot back to the original "Wide" format
df_wide = df_long.pivot(index=['datetime', 'underlying_price'], 
                        columns='Ticker', 
                        values='Final_IV').reset_index()

# 3. Ensure columns are in the exact same order as the original dataset
original_cols = pd.read_csv('dataset.csv').columns
df_wide = df_wide[original_cols]

# 4. Save the baseline submission
df_wide.to_csv('rbf_baseline_submission.csv', index=False)
print("Saved to 'rbf_baseline_submission.csv'. Ready to pass through your converter!")

Saved to 'rbf_baseline_submission.csv'. Ready to pass through your converter!


In [6]:
import pandas as pd
import numpy as np

# 1. Load the original wide dataset
df_orig = pd.read_csv('dataset.csv')

# 2. Extract only the option ticker columns (the numerical matrix)
id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

# Convert to a pure numpy matrix
matrix_orig = df_orig[ticker_cols].values
mask_missing = np.isnan(matrix_orig)

# 3. Initialize the missing values with column means (a sensible starting guess)
col_means = np.nanmean(matrix_orig, axis=0)
matrix_filled = np.where(mask_missing, col_means, matrix_orig)

# 4. Iterative SVD Truncated Matrix Completion (Soft-Impute approach)
# We use rank=2 because Level and Skew capture almost all IV surface structure
rank = 2 
iterations = 15

print(f"Running Iterative SVD completion (Rank={rank}) over {iterations} iterations...")

for i in range(iterations):
    # Compute Singular Value Decomposition
    U, s, Vt = np.linalg.svd(matrix_filled, full_matrices=False)
    
    # Truncate to the top principal components (Rank truncation)
    S = np.diag(s[:rank])
    matrix_reconstructed = np.dot(U[:, :rank], np.dot(S, Vt[:rank, :]))
    
    # Update ONLY the missing values with our reconstruction
    matrix_filled[mask_missing] = matrix_reconstructed[mask_missing]

# 5. Bring the filled matrix back into a clean dataframe
df_svd = df_orig.copy()
df_svd[ticker_cols] = matrix_filled

# Clip to realistic financial bounds just in case
df_svd[ticker_cols] = df_svd[ticker_cols].clip(lower=0.001, upper=3.0)

# 6. Save the SVD baseline submission
df_svd.to_csv('svd_baseline_submission.csv', index=False)
print("\nSuccess! SVD Matrix Completion saved to 'svd_baseline_submission.csv'")

Running Iterative SVD completion (Rank=2) over 15 iterations...

Success! SVD Matrix Completion saved to 'svd_baseline_submission.csv'


In [7]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# 1. Load data and setup
df_long = pd.read_csv('dataset.csv')
id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_long.columns if col not in id_vars]

df_melted = pd.melt(df_long, id_vars=id_vars, value_vars=ticker_cols, 
                  var_name='Ticker', value_name='Implied Volatility')

df_melted['Strike'] = df_melted['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)

# 2. For SVI, we use Log-Moneyness (k) instead of regular moneyness
df_melted['Log_Moneyness'] = np.log(df_melted['Strike'] / df_melted['underlying_price'])

# 3. Define the Raw SVI Equation (Gatheral's formulation)
# w(k) = a + b * (rho * (k - m) + sqrt((k - m)^2 + sigma^2))
def svi_variance(k, a, b, rho, m, sigma):
    return a + b * (rho * (k - m) + np.sqrt((k - m)**2 + sigma**2))

# Objective function to minimize the error between SVI and actual data
def objective(params, k_data, vol_data):
    a, b, rho, m, sigma = params
    # Penalty to keep parameters within theoretical bounds
    if b < 0 or abs(rho) > 1 or sigma <= 0 or a < 0:
        return 1e6 
    
    # SVI fits variance (volatility squared)
    var_data = vol_data ** 2
    var_pred = svi_variance(k_data, a, b, rho, m, sigma)
    return np.mean((var_data - var_pred) ** 2)

# 4. Loop through and fit SVI per timestamp
df_melted['SVI_Pred'] = np.nan
timestamps = df_melted['datetime'].unique()

print(f"Fitting SVI Models across {len(timestamps)} timestamps. This may take a minute...")

for ts in timestamps:
    ts_mask = df_melted['datetime'] == ts
    ts_data = df_melted[ts_mask]
    
    known = ts_data[ts_data['Implied Volatility'].notna()]
    missing = ts_data[ts_data['Implied Volatility'].isna()]
    
    if len(missing) == 0 or len(known) < 5: # Need at least 5 points to fit 5 parameters
        continue
        
    k_train = known['Log_Moneyness'].values
    v_train = known['Implied Volatility'].values
    k_predict = missing['Log_Moneyness'].values
    
    # Initial guess for parameters: [a, b, rho, m, sigma]
    initial_guess = [np.min(v_train)**2, 0.1, 0.0, 0.0, 0.1]
    
    try:
        # Optimize!
        result = minimize(objective, initial_guess, args=(k_train, v_train), method='Nelder-Mead')
        a_opt, b_opt, rho_opt, m_opt, sigma_opt = result.x
        
        # Predict missing variance and convert back to volatility (sqrt)
        var_preds = svi_variance(k_predict, a_opt, b_opt, rho_opt, m_opt, sigma_opt)
        
        # Ensure we don't take square root of negative numbers due to numerical instability
        var_preds = np.maximum(var_preds, 1e-6) 
        vol_preds = np.sqrt(var_preds)
        
        df_melted.loc[missing.index, 'SVI_Pred'] = np.clip(vol_preds, 0.001, 3.0)
        
    except Exception as e:
        pass

# 5. Format back to Wide format for Kaggle
df_melted['Final_IV'] = df_melted['Implied Volatility'].fillna(df_melted['SVI_Pred'])
df_svi_wide = df_melted.pivot(index=['datetime', 'underlying_price'], 
                        columns='Ticker', 
                        values='Final_IV').reset_index()

df_svi_wide = df_svi_wide[df_long.columns]
df_svi_wide.to_csv('svi_baseline_submission.csv', index=False)
print("Success! SVI Predictions saved to 'svi_baseline_submission.csv'")

Fitting SVI Models across 975 timestamps. This may take a minute...
Success! SVI Predictions saved to 'svi_baseline_submission.csv'


In [8]:
import pandas as pd

# 1. Load all three independent baseline submissions
df_rbf = pd.read_csv('rbf_baseline_submission.csv')
df_svd = pd.read_csv('svd_baseline_submission.csv')
df_svi = pd.read_csv('svi_baseline_submission.csv')
df_orig = pd.read_csv('dataset.csv')

# 2. Separate identifiers from ticker columns
id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

# 3. Initialize final submission dataframe
df_final = df_orig.copy()

# 4. Define Tuned Weights (Must add up to 1.0)
# We place higher weight on SVD and SVI because they capture global 
# patterns and financial theory much better than raw geometric RBF.
w_rbf = 0.20
w_svd = 0.40
w_svi = 0.40

print(f"Blending models with weights: RBF={w_rbf}, SVD={w_svd}, SVI={w_svi}...")

# 5. Apply the linear blend across all target columns
df_final[ticker_cols] = (
    (df_rbf[ticker_cols] * w_rbf) + 
    (df_svd[ticker_cols] * w_svd) + 
    (df_svi[ticker_cols] * w_svi)
)

# 6. Final integrity verification check
remaining_nans = df_final[ticker_cols].isna().sum().sum()
print(f"Total missing values left in final matrix: {remaining_nans}")

# 7. Save the master submission file
df_final.to_csv('final_3way_ensemble.csv', index=False)
print("\nBoom! Master submission saved to 'final_3way_ensemble.csv'")

Blending models with weights: RBF=0.2, SVD=0.4, SVI=0.4...
Total missing values left in final matrix: 0

Boom! Master submission saved to 'final_3way_ensemble.csv'


In [9]:
import pandas as pd

# Define your file paths clearly here
ORIGINAL_DATASET_PATH = "dataset.csv" 
FILLED_DATASET_PATH = "final_3way_ensemble.csv"

# --- CHANGE THE NAME HERE TO ANYTHING YOU WANT ---
OUTPUT_SUBMISSION_PATH = "final_leaderboard_submission.csv" 

SEPARATOR = "||"

def generate_solution(filled_path: str, output_path: str):
    # 1. Load the data
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)

    feature_cols = [c for c in original.columns if c != "datetime"]

    rows = []
    print("Extracting missing values and formatting keys...")
    
    # 2. Map wide matrix values back into long leaderboard rows
    for col in feature_cols:
        was_missing = original[col].isna()

        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    # 3. Create the clean submission DataFrame
    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)
    
    # 4. Save to disk with the new name
    solution.to_csv(output_path, index=False)
    print(f"\n✅ Solution saved → {output_path}  ({len(solution)} rows)")

# Execute the function
generate_solution(FILLED_DATASET_PATH, OUTPUT_SUBMISSION_PATH)

Extracting missing values and formatting keys...

✅ Solution saved → final_leaderboard_submission.csv  (5460 rows)


In [10]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# 1. Load Original (with NaNs) and Ensemble (Filled) datasets
df_orig = pd.read_csv('dataset.csv')
df_ens = pd.read_csv('final_3way_ensemble.csv')

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

# Melt both to long format so we can train a machine learning model on the rows
df_orig_long = pd.melt(df_orig, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='True_IV')
df_ens_long = pd.melt(df_ens, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='Ensemble_IV')

# Merge them together
df = pd.merge(df_orig_long, df_ens_long, on=['datetime', 'underlying_price', 'Ticker'])

# 2. Extract Features from Ticker (e.g., NIFTY27JAN2625200CE)
print("Extracting Time-to-Maturity (TTM) and Moneyness features...")
df['Strike'] = df['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df['Moneyness'] = df['Strike'] / df['underlying_price']

# Extract Call vs Put (1 for Call, 0 for Put)
df['Is_Call'] = (df['Ticker'].str[-2:] == 'CE').astype(int)

# Extract Expiry Date (Grabs the 27JAN26 part of the string)
expiry_str = df['Ticker'].str.extract(r'NIFTY(\d{2}[A-Z]{3}\d{2})')[0]
df['Expiry_Date'] = pd.to_datetime(expiry_str, format='%d%b%y')
df['Current_Date'] = pd.to_datetime(df['datetime'])

# Calculate TTM in years
df['TTM'] = (df['Expiry_Date'] - df['Current_Date']).dt.total_seconds() / (365.25 * 24 * 3600)
# Clip at a minimum of 1 day to prevent 0 or negative values on expiration day
df['TTM'] = df['TTM'].clip(lower=1/365.25)

# 3. Calculate Residuals (The tiny errors our ensemble made on the known data)
known_mask = df['True_IV'].notna()
missing_mask = df['True_IV'].isna()

df['Residual'] = np.nan
df.loc[known_mask, 'Residual'] = df.loc[known_mask, 'True_IV'] - df.loc[known_mask, 'Ensemble_IV']

# 4. Train LightGBM on the Residuals
print("Training LightGBM Regressor on Ensemble Residuals...")
features = ['Moneyness', 'TTM', 'Is_Call', 'Ensemble_IV']

X_train = df.loc[known_mask, features]
y_train = df.loc[known_mask, 'Residual']
X_predict = df.loc[missing_mask, features]

# Hyperparameters tuned for residual learning (low learning rate, shallow trees to prevent overfitting)
model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# 5. Predict Residuals for Missing Data & Apply the micro-corrections
predicted_residuals = model.predict(X_predict)
df.loc[missing_mask, 'Final_Boosted_IV'] = df.loc[missing_mask, 'Ensemble_IV'] + predicted_residuals

# Ensure we keep the originally known data perfectly intact
df.loc[known_mask, 'Final_Boosted_IV'] = df.loc[known_mask, 'True_IV']

# Clip to realistic financial bounds just in case LightGBM overshoots
df['Final_Boosted_IV'] = df['Final_Boosted_IV'].clip(lower=0.001, upper=3.0)

# 6. Pivot back to Wide Format for your Converter
print("Pivoting back to master format...")
df_boosted_wide = df.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='Final_Boosted_IV').reset_index()
df_boosted_wide = df_boosted_wide[df_orig.columns]

# Save to disk
df_boosted_wide.to_csv('lgbm_boosted_ensemble.csv', index=False)
print("\n✅ Success! LightGBM Boosted matrix saved to 'lgbm_boosted_ensemble.csv'")

Extracting Time-to-Maturity (TTM) and Moneyness features...


ValueError: time data "13-01-2026 09:15" doesn't match format "%m-%d-%Y %H:%M". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [11]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# 1. Load Original (with NaNs) and Ensemble (Filled) datasets
df_orig = pd.read_csv('dataset.csv')
df_ens = pd.read_csv('final_3way_ensemble.csv')

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

# Melt both to long format so we can train a machine learning model on the rows
df_orig_long = pd.melt(df_orig, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='True_IV')
df_ens_long = pd.melt(df_ens, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='Ensemble_IV')

# Merge them together
df = pd.merge(df_orig_long, df_ens_long, on=['datetime', 'underlying_price', 'Ticker'])

# 2. Extract Features from Ticker (e.g., NIFTY27JAN2625200CE)
print("Extracting Time-to-Maturity (TTM) and Moneyness features...")
df['Strike'] = df['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df['Moneyness'] = df['Strike'] / df['underlying_price']

# Extract Call vs Put (1 for Call, 0 for Put)
df['Is_Call'] = (df['Ticker'].str[-2:] == 'CE').astype(int)

# Extract Expiry Date (Grabs the 27JAN26 part of the string)
expiry_str = df['Ticker'].str.extract(r'NIFTY(\d{2}[A-Z]{3}\d{2})')[0]
df['Expiry_Date'] = pd.to_datetime(expiry_str, format='%d%b%y')

# --- FIX: Explicitly passing the exact format to handle DD-MM-YYYY HH:MM ---
df['Current_Date'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M')

# Calculate TTM in years
df['TTM'] = (df['Expiry_Date'] - df['Current_Date']).dt.total_seconds() / (365.25 * 24 * 3600)
# Clip at a minimum of 1 day to prevent 0 or negative values on expiration day
df['TTM'] = df['TTM'].clip(lower=1/365.25)

# 3. Calculate Residuals (The tiny errors our ensemble made on the known data)
known_mask = df['True_IV'].notna()
missing_mask = df['True_IV'].isna()

df['Residual'] = np.nan
df.loc[known_mask, 'Residual'] = df.loc[known_mask, 'True_IV'] - df.loc[known_mask, 'Ensemble_IV']

# 4. Train LightGBM on the Residuals
print("Training LightGBM Regressor on Ensemble Residuals...")
features = ['Moneyness', 'TTM', 'Is_Call', 'Ensemble_IV']

X_train = df.loc[known_mask, features]
y_train = df.loc[known_mask, 'Residual']
X_predict = df.loc[missing_mask, features]

# Hyperparameters tuned for residual learning
model = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# 5. Predict Residuals for Missing Data & Apply the micro-corrections
predicted_residuals = model.predict(X_predict)
df.loc[missing_mask, 'Final_Boosted_IV'] = df.loc[missing_mask, 'Ensemble_IV'] + predicted_residuals

# Ensure we keep the originally known data perfectly intact
df.loc[known_mask, 'Final_Boosted_IV'] = df.loc[known_mask, 'True_IV']

# Clip to realistic financial bounds just in case LightGBM overshoots
df['Final_Boosted_IV'] = df['Final_Boosted_IV'].clip(lower=0.001, upper=3.0)

# 6. Pivot back to Wide Format for your Converter
print("Pivoting back to master format...")
df_boosted_wide = df.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='Final_Boosted_IV').reset_index()
df_boosted_wide = df_boosted_wide[df_orig.columns]

# Save to disk
df_boosted_wide.to_csv('lgbm_boosted_ensemble.csv', index=False)
print("\n✅ Success! LightGBM Boosted matrix saved to 'lgbm_boosted_ensemble.csv'")

Extracting Time-to-Maturity (TTM) and Moneyness features...
Training LightGBM Regressor on Ensemble Residuals...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000806 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 766
[LightGBM] [Info] Number of data points in the train set: 21840, number of used features: 4
[LightGBM] [Info] Start training from score 0.000195
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

In [12]:
import pandas as pd

# Define your file paths clearly here
ORIGINAL_DATASET_PATH = "dataset.csv" 

# --- WE ARE NOW USING YOUR NEW LIGHTGBM MATRIX ---
FILLED_DATASET_PATH = "lgbm_boosted_ensemble.csv"

# --- THE FINAL SUBMISSION FILE ---
OUTPUT_SUBMISSION_PATH = "ultimate_lgbm_submission.csv" 

SEPARATOR = "||"

def generate_solution(filled_path: str, output_path: str):
    # 1. Load the data
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)

    feature_cols = [c for c in original.columns if c != "datetime"]

    rows = []
    print("Extracting LightGBM boosted values and formatting keys...")
    
    # 2. Map wide matrix values back into long leaderboard rows
    for col in feature_cols:
        was_missing = original[col].isna()

        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    # 3. Create the clean submission DataFrame
    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)
    
    # 4. Save to disk with the new name
    solution.to_csv(output_path, index=False)
    print(f"\n✅ Solution saved → {output_path}  ({len(solution)} rows)")

# Execute the function
generate_solution(FILLED_DATASET_PATH, OUTPUT_SUBMISSION_PATH)

Extracting LightGBM boosted values and formatting keys...

✅ Solution saved → ultimate_lgbm_submission.csv  (5460 rows)


In [13]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

print("Step 1: Loading all underlying model baselines...")
df_orig = pd.read_csv('dataset.csv')
df_rbf  = pd.read_csv('rbf_baseline_submission.csv')
df_svd  = pd.read_csv('svd_baseline_submission.csv')
df_svi  = pd.read_csv('svi_baseline_submission.csv')

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

# Melt everything to long format to build our Meta-Dataset
print("Reshaping matrices for stacking...")
m_orig = pd.melt(df_orig, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='True_IV')
m_rbf  = pd.melt(df_rbf,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='RBF_IV')
m_svd  = pd.melt(df_svd,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVD_IV')
m_svi  = pd.melt(df_svi,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVI_IV')

# Merge all models into a single master dataframe
df_meta = m_orig
df_meta['RBF_IV'] = m_rbf['RBF_IV']
df_meta['SVD_IV'] = m_svd['SVD_IV']
df_meta['SVI_IV'] = m_svi['SVI_IV']

# Step 2: Advanced Feature Engineering
print("Engineering domain-specific options features...")
df_meta['Strike'] = df_meta['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df_meta['Moneyness'] = df_meta['Strike'] / df_meta['underlying_price']
df_meta['Log_Moneyness'] = np.log(df_meta['underlying_price'] / df_meta['Strike'])
df_meta['Is_Call'] = (df_meta['Ticker'].str[-2:] == 'CE').astype(int)

# Extract Expiry and calculate precise TTM
expiry_str = df_meta['Ticker'].str.extract(r'NIFTY(\d{2}[A-Z]{3}\d{2})')[0]
df_meta['Expiry_Date'] = pd.to_datetime(expiry_str, format='%d%b%y')
df_meta['Current_Date'] = pd.to_datetime(df_meta['datetime'], format='%d-%m-%Y %H:%M')
df_meta['TTM'] = (df_meta['Expiry_Date'] - df_meta['Current_Date']).dt.total_seconds() / (365.25 * 24 * 3600)
df_meta['TTM'] = df_meta['TTM'].clip(lower=1/365.25)

# Volatility Regime Tracking: Calculate the average known IV per timestamp to guide the model
vix_proxy = df_meta.groupby('datetime')['True_IV'].transform('mean')
# Fill missing timestamps' proxies with the global mean
df_meta['Vol_Regime'] = vix_proxy.fillna(df_meta['True_IV'].mean())

# Step 3: Performance Audit
known_mask = df_meta['True_IV'].notna()
missing_mask = df_meta['True_IV'].isna()

print("\n--- INDIVIDUAL MODEL PERFORMANCE AUDIT (MSE) ---")
for col in ['RBF_IV', 'SVD_IV', 'SVI_IV']:
    mse = np.mean((df_meta.loc[known_mask, 'True_IV'] - df_meta.loc[known_mask, col])**2)
    print(f"-> {col} Base MSE: {mse:.7f}")

# Step 4: Mathematical Optimization of Weights (Scipy)
def mse_loss(weights):
    w1, w2, w3 = weights
    pred = w1 * df_meta.loc[known_mask, 'RBF_IV'] + w2 * df_meta.loc[known_mask, 'SVD_IV'] + w3 * df_meta.loc[known_mask, 'SVI_IV']
    return np.mean((df_meta.loc[known_mask, 'True_IV'] - pred)**2)

cons = ({'type': 'eq', 'fun': lambda w: 1 - sum(w)})
bounds = [(0, 1), (0, 1), (0, 1)]
res = minimize(mse_loss, [0.33, 0.33, 0.33], method='SLSQP', bounds=bounds, constraints=cons)
w_opt = res.x
print(f"Optimal Mathematical Blend Weights -> RBF: {w_opt[0]:.3f}, SVD: {w_opt[1]:.3f}, SVI: {w_opt[2]:.3f}")
print(f"Optimized Linear Blend MSE: {res.fun:.7f}")

# Step 5: Direct LightGBM Meta-Stacking
print("\nTraining Non-Linear LightGBM Meta-Learner...")
features = ['RBF_IV', 'SVD_IV', 'SVI_IV', 'Moneyness', 'Log_Moneyness', 'TTM', 'Is_Call', 'Vol_Regime']

X_train = df_meta.loc[known_mask, features]
y_train = df_meta.loc[known_mask, 'True_IV']
X_predict = df_meta.loc[missing_mask, features]

meta_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=6,
    num_leaves=45,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
meta_model.fit(X_train, y_train)

# Generate final predictions
df_meta['Meta_Predicted_IV'] = df_meta['True_IV'] # Keep known static
df_meta.loc[missing_mask, 'Meta_Predicted_IV'] = meta_model.predict(X_predict)
df_meta['Meta_Predicted_IV'] = df_meta['Meta_Predicted_IV'].clip(lower=0.001, upper=3.0)

# Step 6: Reconstruction back to Wide Format
print("Reconstructing final matrix...")
df_final_wide = df_meta.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='Meta_Predicted_IV').reset_index()
df_final_wide = df_final_wide[df_orig.columns]

df_final_wide.to_csv('meta_stacked_ensemble.csv', index=False)
print("\n✅ Super-Ensemble matrix successfully saved to 'meta_stacked_ensemble.csv'")

Step 1: Loading all underlying model baselines...
Reshaping matrices for stacking...
Engineering domain-specific options features...

--- INDIVIDUAL MODEL PERFORMANCE AUDIT (MSE) ---
-> RBF_IV Base MSE: 0.0000000
-> SVD_IV Base MSE: 0.0007199
-> SVI_IV Base MSE: 0.0000000
Optimal Mathematical Blend Weights -> RBF: 0.334, SVD: 0.333, SVI: 0.334
Optimized Linear Blend MSE: 0.0000798

Training Non-Linear LightGBM Meta-Learner...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000841 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1786
[LightGBM] [Info] Number of data points in the train set: 21840, number of used features: 8
[LightGBM] [Info] Start training from score 0.182937
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Lig

In [14]:
import pandas as pd

# Define your file paths
ORIGINAL_DATASET_PATH = "dataset.csv" 

# --- USING YOUR NEW META-STACKED MATRIX ---
FILLED_DATASET_PATH = "meta_stacked_ensemble.csv"

# --- THE FINAL SUBMISSION FILE ---
OUTPUT_SUBMISSION_PATH = "ultimate_meta_submission.csv" 

SEPARATOR = "||"

def generate_solution(filled_path: str, output_path: str):
    # 1. Load the data
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)

    feature_cols = [c for c in original.columns if c != "datetime"]

    rows = []
    print("Extracting Meta-Stacked values and formatting keys...")
    
    # 2. Map wide matrix values back into long leaderboard rows
    for col in feature_cols:
        was_missing = original[col].isna()

        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    # 3. Create the clean submission DataFrame
    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)
    
    # 4. Save to disk
    solution.to_csv(output_path, index=False)
    print(f"\n✅ Solution saved → {output_path}  ({len(solution)} rows)")

# Execute the function
generate_solution(FILLED_DATASET_PATH, OUTPUT_SUBMISSION_PATH)

Extracting Meta-Stacked values and formatting keys...

✅ Solution saved → ultimate_meta_submission.csv  (5460 rows)


In [15]:
!pip install xgboost catboost

^C


   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 3.4 MB/s eta 0:00:30
    --------------------------------------- 1.3/101.7 MB 3.3 MB/s eta 0:00:31
    --------------------------------------- 1.6/101.7 MB 3.1 MB/s eta 0:00:33
    --------------------------------------- 1.8/101.7 MB 2.6 MB/s eta 0:00:39
    --------------------------------------- 2.1/101.7 MB 2.1 MB/s eta 0:00:47
    --------------------------------------- 2.4/101.7 MB 1.9 MB/s eta 0:00:52
   - -------------------------------------- 2.6/101.7 MB 1.9 MB/s eta 0:00:52
   - -------------------------------------- 3.1/101.7 MB 1.9 MB/s eta 0:00:53
   - -------------------------------------- 3.4/101.7 MB 1.9 MB/s eta 0:00:53
   - -------------------------------------- 3.9/101.7 MB 1.9 MB/s eta 0:00:52
   - -------------------------------------- 4.2/101.7 MB 1.9 MB/s eta 0:00:52
   - -------------------------------------- 4.5/101.7 MB 1.8 MB/s eta 0

In [16]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

print("Step 1: Loading baseline matrices...")
df_orig = pd.read_csv('dataset.csv')
df_rbf  = pd.read_csv('rbf_baseline_submission.csv')
df_svd  = pd.read_csv('svd_baseline_submission.csv')
df_svi  = pd.read_csv('svi_baseline_submission.csv')

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

print("Reshaping and aligning datasets...")
m_orig = pd.melt(df_orig, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='True_IV')
m_rbf  = pd.melt(df_rbf,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='RBF_IV')
m_svd  = pd.melt(df_svd,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVD_IV')
m_svi  = pd.melt(df_svi,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVI_IV')

df_meta = m_orig
df_meta['RBF_IV'] = m_rbf['RBF_IV']
df_meta['SVD_IV'] = m_svd['SVD_IV']
df_meta['SVI_IV'] = m_svi['SVI_IV']

print("Step 2: Engineering Options Features...")
df_meta['Strike'] = df_meta['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df_meta['Moneyness'] = df_meta['Strike'] / df_meta['underlying_price']
df_meta['Log_Moneyness'] = np.log(df_meta['underlying_price'] / df_meta['Strike'])
df_meta['Is_Call'] = (df_meta['Ticker'].str[-2:] == 'CE').astype(int)

# Exact Time to Maturity (TTM)
expiry_str = df_meta['Ticker'].str.extract(r'NIFTY(\d{2}[A-Z]{3}\d{2})')[0]
df_meta['Expiry_Date'] = pd.to_datetime(expiry_str, format='%d%b%y')
df_meta['Current_Date'] = pd.to_datetime(df_meta['datetime'], format='%d-%m-%Y %H:%M')
df_meta['TTM'] = (df_meta['Expiry_Date'] - df_meta['Current_Date']).dt.total_seconds() / (365.25 * 24 * 3600)
df_meta['TTM'] = df_meta['TTM'].clip(lower=1/365.25)

# Volatility Regime
vix_proxy = df_meta.groupby('datetime')['True_IV'].transform('mean')
df_meta['Vol_Regime'] = vix_proxy.fillna(df_meta['True_IV'].mean())

# --- CRITICAL TARGET TRANSFORMATION: IV to Variance ---
print("Applying Target Transformation (IV -> Variance)...")
df_meta['True_Variance'] = df_meta['True_IV'] ** 2
df_meta['RBF_Var'] = df_meta['RBF_IV'] ** 2
df_meta['SVD_Var'] = df_meta['SVD_IV'] ** 2
df_meta['SVI_Var'] = df_meta['SVI_IV'] ** 2

known_mask = df_meta['True_IV'].notna()
missing_mask = df_meta['True_IV'].isna()

features = ['RBF_Var', 'SVD_Var', 'SVI_Var', 'Moneyness', 'Log_Moneyness', 'TTM', 'Is_Call', 'Vol_Regime']

X_train = df_meta.loc[known_mask, features]
y_train = df_meta.loc[known_mask, 'True_Variance'] # Training on VARIANCE
X_predict = df_meta.loc[missing_mask, features]

print("\nStep 3: Training the Kaggle Trinity Models...")
# 1. LightGBM
print("-> Training LightGBM...")
model_lgb = lgb.LGBMRegressor(n_estimators=600, learning_rate=0.03, max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
model_lgb.fit(X_train, y_train)
pred_lgb = model_lgb.predict(X_predict)

# 2. XGBoost
print("-> Training XGBoost...")
model_xgb = xgb.XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=6, random_state=42, n_jobs=-1)
model_xgb.fit(X_train, y_train)
pred_xgb = model_xgb.predict(X_predict)

# 3. CatBoost (King of tabular data)
print("-> Training CatBoost...")
model_cat = CatBoostRegressor(iterations=600, learning_rate=0.03, depth=6, random_seed=42, verbose=0)
model_cat.fit(X_train, y_train)
pred_cat = model_cat.predict(X_predict)

print("\nStep 4: Blending Variance Predictions & Reversing Transformation...")
# Blend the variance predictions equally
ensemble_variance_pred = (pred_lgb + pred_xgb + pred_cat) / 3.0

# Reverse the transformation (Square Root of Variance = Volatility)
# np.maximum prevents negative square roots if models predicted tiny negative numbers
final_iv_pred = np.sqrt(np.maximum(ensemble_variance_pred, 1e-7))

df_meta['Final_Predicted_IV'] = df_meta['True_IV']
df_meta.loc[missing_mask, 'Final_Predicted_IV'] = final_iv_pred
df_meta['Final_Predicted_IV'] = df_meta['Final_Predicted_IV'].clip(lower=0.001, upper=3.0)

print("Step 5: Saving the Master Matrix...")
df_final_wide = df_meta.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='Final_Predicted_IV').reset_index()
df_final_wide = df_final_wide[df_orig.columns]

df_final_wide.to_csv('trinity_variance_ensemble.csv', index=False)
print("\n✅ Success! Trinity Model saved to 'trinity_variance_ensemble.csv'")

Step 1: Loading baseline matrices...
Reshaping and aligning datasets...
Step 2: Engineering Options Features...
Applying Target Transformation (IV -> Variance)...

Step 3: Training the Kaggle Trinity Models...
-> Training LightGBM...
-> Training XGBoost...
-> Training CatBoost...

Step 4: Blending Variance Predictions & Reversing Transformation...
Step 5: Saving the Master Matrix...

✅ Success! Trinity Model saved to 'trinity_variance_ensemble.csv'


In [17]:
import pandas as pd

# Define your file paths
ORIGINAL_DATASET_PATH = "dataset.csv" 

# --- USING YOUR NEW TRINITY VARIANCE MATRIX ---
FILLED_DATASET_PATH = "trinity_variance_ensemble.csv"

# --- THE FINAL SUBMISSION FILE ---
OUTPUT_SUBMISSION_PATH = "trinity_final_submission.csv" 

SEPARATOR = "||"

def generate_solution(filled_path: str, output_path: str):
    # 1. Load the data
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)

    feature_cols = [c for c in original.columns if c != "datetime"]

    rows = []
    print("Extracting Trinity Variance values and formatting keys for Kaggle...")
    
    # 2. Map wide matrix values back into long leaderboard rows
    for col in feature_cols:
        was_missing = original[col].isna()

        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    # 3. Create the clean submission DataFrame
    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)
    
    # 4. Save to disk
    solution.to_csv(output_path, index=False)
    print(f"\n✅ Solution saved → {output_path}  ({len(solution)} rows)")

# Execute the function
generate_solution(FILLED_DATASET_PATH, OUTPUT_SUBMISSION_PATH)

Extracting Trinity Variance values and formatting keys for Kaggle...

✅ Solution saved → trinity_final_submission.csv  (5460 rows)


In [18]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("Step 1: Loading baseline matrices...")
df_orig = pd.read_csv('dataset.csv')
df_rbf  = pd.read_csv('rbf_baseline_submission.csv')
df_svd  = pd.read_csv('svd_baseline_submission.csv')
df_svi  = pd.read_csv('svi_baseline_submission.csv')

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

print("Step 2: Reshaping and extracting features...")
m_orig = pd.melt(df_orig, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='True_IV')
m_rbf  = pd.melt(df_rbf,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='RBF_IV')
m_svd  = pd.melt(df_svd,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVD_IV')
m_svi  = pd.melt(df_svi,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVI_IV')

df_meta = m_orig.copy()
df_meta['RBF_IV'] = m_rbf['RBF_IV']
df_meta['SVD_IV'] = m_svd['SVD_IV']
df_meta['SVI_IV'] = m_svi['SVI_IV']

df_meta['Strike'] = df_meta['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df_meta['Is_Call'] = (df_meta['Ticker'].str[-2:] == 'CE').astype(int)

# Calculate distance from the underlying price
df_meta['Log_Moneyness'] = np.log(df_meta['underlying_price'] / df_meta['Strike'])
df_meta['Abs_Distance'] = np.abs(df_meta['Log_Moneyness'])

# Segment into 3 distinct Market Regimes based on distance
cond_atm = df_meta['Abs_Distance'] <= 0.02
cond_ntm = (df_meta['Abs_Distance'] > 0.02) & (df_meta['Abs_Distance'] <= 0.06)
cond_far = df_meta['Abs_Distance'] > 0.06

df_meta['Regime'] = np.select([cond_atm, cond_ntm, cond_far], ['ATM', 'NTM', 'FAR'])

features = ['RBF_IV', 'SVD_IV', 'SVI_IV', 'Log_Moneyness', 'Abs_Distance', 'Is_Call']
df_meta['Final_Predicted_IV'] = df_meta['True_IV']

known_mask = df_meta['True_IV'].notna()
missing_mask = df_meta['True_IV'].isna()

print("\nStep 3: Training Market Regime Meta-Learners...")
regimes = ['ATM', 'NTM', 'FAR']

for regime in regimes:
    print(f"-> Training specialized LightGBM for {regime} options...")
    regime_mask = df_meta['Regime'] == regime
    
    train_idx = known_mask & regime_mask
    pred_idx = missing_mask & regime_mask
    
    X_train = df_meta.loc[train_idx, features]
    y_train = df_meta.loc[train_idx, 'True_IV']
    X_predict = df_meta.loc[pred_idx, features]
    
    if len(X_predict) > 0:
        # Fine-tuned hyper-parameters to prevent overfitting on the smaller segmented datasets
        model = lgb.LGBMRegressor(
            n_estimators=800, 
            learning_rate=0.01, 
            max_depth=6, 
            num_leaves=25, 
            random_state=42, 
            n_jobs=-1, 
            verbose=-1
        )
        model.fit(X_train, y_train)
        
        preds = model.predict(X_predict)
        df_meta.loc[pred_idx, 'Final_Predicted_IV'] = preds

print("\nStep 4: Reconstructing Master Matrix...")
df_meta['Final_Predicted_IV'] = df_meta['Final_Predicted_IV'].clip(lower=0.001, upper=3.0)

df_final_wide = df_meta.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='Final_Predicted_IV').reset_index()
df_final_wide = df_final_wide[df_orig.columns]

df_final_wide.to_csv('regime_meta_ensemble.csv', index=False)
print("✅ Success! Regime Segmentation Matrix saved to 'regime_meta_ensemble.csv'")

Step 1: Loading baseline matrices...
Step 2: Reshaping and extracting features...


TypeError: Choicelist and default value do not have a common dtype: The DType <class 'numpy.dtypes._PyLongDType'> could not be promoted by <class 'numpy.dtypes.StrDType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes.StrDType'>, <class 'numpy.dtypes._PyLongDType'>)

In [19]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("Step 1: Loading baseline matrices...")
df_orig = pd.read_csv('dataset.csv')
df_rbf  = pd.read_csv('rbf_baseline_submission.csv')
df_svd  = pd.read_csv('svd_baseline_submission.csv')
df_svi  = pd.read_csv('svi_baseline_submission.csv')

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

print("Step 2: Reshaping and extracting features...")
m_orig = pd.melt(df_orig, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='True_IV')
m_rbf  = pd.melt(df_rbf,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='RBF_IV')
m_svd  = pd.melt(df_svd,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVD_IV')
m_svi  = pd.melt(df_svi,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVI_IV')

df_meta = m_orig.copy()
df_meta['RBF_IV'] = m_rbf['RBF_IV']
df_meta['SVD_IV'] = m_svd['SVD_IV']
df_meta['SVI_IV'] = m_svi['SVI_IV']

df_meta['Strike'] = df_meta['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df_meta['Is_Call'] = (df_meta['Ticker'].str[-2:] == 'CE').astype(int)

# Calculate distance from the underlying price
df_meta['Log_Moneyness'] = np.log(df_meta['underlying_price'] / df_meta['Strike'])
df_meta['Abs_Distance'] = np.abs(df_meta['Log_Moneyness'])

# Segment into 3 distinct Market Regimes based on distance
cond_atm = df_meta['Abs_Distance'] <= 0.02
cond_ntm = (df_meta['Abs_Distance'] > 0.02) & (df_meta['Abs_Distance'] <= 0.06)
cond_far = df_meta['Abs_Distance'] > 0.06

# --- FIXED LINE: Added default='UNKNOWN' to prevent the TypeError ---
df_meta['Regime'] = np.select([cond_atm, cond_ntm, cond_far], ['ATM', 'NTM', 'FAR'], default='UNKNOWN')

features = ['RBF_IV', 'SVD_IV', 'SVI_IV', 'Log_Moneyness', 'Abs_Distance', 'Is_Call']
df_meta['Final_Predicted_IV'] = df_meta['True_IV']

known_mask = df_meta['True_IV'].notna()
missing_mask = df_meta['True_IV'].isna()

print("\nStep 3: Training Market Regime Meta-Learners...")
regimes = ['ATM', 'NTM', 'FAR']

for regime in regimes:
    print(f"-> Training specialized LightGBM for {regime} options...")
    regime_mask = df_meta['Regime'] == regime
    
    train_idx = known_mask & regime_mask
    pred_idx = missing_mask & regime_mask
    
    X_train = df_meta.loc[train_idx, features]
    y_train = df_meta.loc[train_idx, 'True_IV']
    X_predict = df_meta.loc[pred_idx, features]
    
    if len(X_predict) > 0:
        # Fine-tuned hyper-parameters to prevent overfitting on the smaller segmented datasets
        model = lgb.LGBMRegressor(
            n_estimators=800, 
            learning_rate=0.01, 
            max_depth=6, 
            num_leaves=25, 
            random_state=42, 
            n_jobs=-1, 
            verbose=-1
        )
        model.fit(X_train, y_train)
        
        preds = model.predict(X_predict)
        df_meta.loc[pred_idx, 'Final_Predicted_IV'] = preds

print("\nStep 4: Reconstructing Master Matrix...")
df_meta['Final_Predicted_IV'] = df_meta['Final_Predicted_IV'].clip(lower=0.001, upper=3.0)

df_final_wide = df_meta.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='Final_Predicted_IV').reset_index()
df_final_wide = df_final_wide[df_orig.columns]

df_final_wide.to_csv('regime_meta_ensemble.csv', index=False)
print("✅ Success! Regime Segmentation Matrix saved to 'regime_meta_ensemble.csv'")

Step 1: Loading baseline matrices...
Step 2: Reshaping and extracting features...

Step 3: Training Market Regime Meta-Learners...
-> Training specialized LightGBM for ATM options...
-> Training specialized LightGBM for NTM options...
-> Training specialized LightGBM for FAR options...

Step 4: Reconstructing Master Matrix...
✅ Success! Regime Segmentation Matrix saved to 'regime_meta_ensemble.csv'


In [20]:
import pandas as pd

# Define your file paths
ORIGINAL_DATASET_PATH = "dataset.csv" 

# --- USING YOUR NEW REGIME SEGMENTATION MATRIX ---
FILLED_DATASET_PATH = "regime_meta_ensemble.csv"

# --- THE FINAL SUBMISSION FILE ---
OUTPUT_SUBMISSION_PATH = "regime_final_submission.csv" 

SEPARATOR = "||"

def generate_solution(filled_path: str, output_path: str):
    # 1. Load the original dataset and your newly filled regime matrix
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)

    feature_cols = [c for c in original.columns if c != "datetime"]

    rows = []
    print("Extracting Regime-Segmented values and formatting keys for Kaggle...")
    
    # 2. Map wide matrix values back into long leaderboard rows
    for col in feature_cols:
        was_missing = original[col].isna()

        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    # 3. Create the clean submission DataFrame
    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)
    
    # 4. Save to disk
    solution.to_csv(output_path, index=False)
    print(f"\n✅ Solution saved → {output_path}  ({len(solution)} rows)")

# Execute the function
generate_solution(FILLED_DATASET_PATH, OUTPUT_SUBMISSION_PATH)

Extracting Regime-Segmented values and formatting keys for Kaggle...

✅ Solution saved → regime_final_submission.csv  (5460 rows)


In [21]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print("Step 1: Loading matrices...")
df_orig = pd.read_csv('dataset.csv')
df_rbf  = pd.read_csv('rbf_baseline_submission.csv')
df_svd  = pd.read_csv('svd_baseline_submission.csv')
df_svi  = pd.read_csv('svi_baseline_submission.csv')

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

print("Step 2: Reshaping and extracting continuous features...")
m_orig = pd.melt(df_orig, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='True_IV')
m_rbf  = pd.melt(df_rbf,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='RBF_IV')
m_svd  = pd.melt(df_svd,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVD_IV')
m_svi  = pd.melt(df_svi,  id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='SVI_IV')

df_meta = m_orig.copy()
df_meta['RBF_IV'] = m_rbf['RBF_IV']
df_meta['SVD_IV'] = m_svd['SVD_IV']
df_meta['SVI_IV'] = m_svi['SVI_IV']

# Feature Engineering
df_meta['Strike'] = df_meta['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df_meta['Is_Call'] = (df_meta['Ticker'].str[-2:] == 'CE').astype(int)
df_meta['Log_Moneyness'] = np.log(df_meta['underlying_price'] / df_meta['Strike'])

# TTM Extraction
expiry_str = df_meta['Ticker'].str.extract(r'NIFTY(\d{2}[A-Z]{3}\d{2})')[0]
df_meta['Expiry_Date'] = pd.to_datetime(expiry_str, format='%d%b%y')
df_meta['Current_Date'] = pd.to_datetime(df_meta['datetime'], format='%d-%m-%Y %H:%M')
df_meta['TTM'] = (df_meta['Expiry_Date'] - df_meta['Current_Date']).dt.total_seconds() / (365.25 * 24 * 3600)
df_meta['TTM'] = df_meta['TTM'].clip(lower=1/365.25)

features = ['RBF_IV', 'SVD_IV', 'SVI_IV', 'Log_Moneyness', 'TTM', 'Is_Call']

known_mask = df_meta['True_IV'].notna()
missing_mask = df_meta['True_IV'].isna()

X_train = df_meta.loc[known_mask, features]
y_train = df_meta.loc[known_mask, 'True_IV']
X_predict = df_meta.loc[missing_mask, features]

print("Step 3: Scaling features for Neural Network...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_predict_scaled = scaler.transform(X_predict)

print("Step 4: Training Smooth MLP Neural Network...")
# A moderately deep network that enforces smooth interpolation without overfitting
nn_model = MLPRegressor(
    hidden_layer_sizes=(64, 32), 
    activation='relu',           
    solver='adam',               
    alpha=0.01,                  # L2 penalty (Ridge regularization) to guarantee smoothness
    learning_rate_init=0.005,
    max_iter=500,
    random_state=42,
    early_stopping=True
)

nn_model.fit(X_train_scaled, y_train)

print(f"Neural Network Training Score (R^2): {nn_model.score(X_train_scaled, y_train):.5f}")

print("Step 5: Generating smooth predictions...")
df_meta['Final_Predicted_IV'] = df_meta['True_IV']
df_meta.loc[missing_mask, 'Final_Predicted_IV'] = nn_model.predict(X_predict_scaled)

# Ensure no negative IVs
df_meta['Final_Predicted_IV'] = df_meta['Final_Predicted_IV'].clip(lower=0.001, upper=3.0)

df_final_wide = df_meta.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='Final_Predicted_IV').reset_index()
df_final_wide = df_final_wide[df_orig.columns]

df_final_wide.to_csv('neural_net_ensemble.csv', index=False)
print("✅ Success! Smooth Neural Network Matrix saved to 'neural_net_ensemble.csv'")

Step 1: Loading matrices...
Step 2: Reshaping and extracting continuous features...
Step 3: Scaling features for Neural Network...
Step 4: Training Smooth MLP Neural Network...
Neural Network Training Score (R^2): 0.99986
Step 5: Generating smooth predictions...
✅ Success! Smooth Neural Network Matrix saved to 'neural_net_ensemble.csv'


In [22]:
import pandas as pd

ORIGINAL_DATASET_PATH = "dataset.csv" 
FILLED_DATASET_PATH = "neural_net_ensemble.csv"
OUTPUT_SUBMISSION_PATH = "nn_final_submission.csv" 
SEPARATOR = "||"

def generate_solution(filled_path: str, output_path: str):
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)
    feature_cols = [c for c in original.columns if c != "datetime"]
    rows = []
    
    for col in feature_cols:
        was_missing = original[col].isna()
        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)
    solution.to_csv(output_path, index=False)
    print(f"\n✅ Solution saved → {output_path}  ({len(solution)} rows)")

generate_solution(FILLED_DATASET_PATH, OUTPUT_SUBMISSION_PATH)


✅ Solution saved → nn_final_submission.csv  (5460 rows)


In [23]:
import pandas as pd
import numpy as np

print("Step 1: Loading Original Dataset...")
df = pd.read_csv('dataset.csv')

# Crucial: Convert to true datetime objects and sort chronologically
df['datetime'] = pd.to_datetime(df['datetime'], format='%d-%m-%Y %H:%M')
df = df.sort_values('datetime')

# Set datetime as the index so Pandas knows the exact time distances
df = df.set_index('datetime')

# Separate the underlying price so we only interpolate the options
underlying = df['underlying_price']
df_iv = df.drop(columns=['underlying_price'])

print("Step 2: Applying Strict Time-Series Interpolation...")
# method='time' mathematically calculates the exact time gap (e.g. 5 mins vs 1 hour)
# and interpolates the missing IVs perfectly along that timeline.
df_iv_filled = df_iv.interpolate(method='time', limit_direction='both')

# Step 3: Fallback (Just in case a ticker was missing for the ENTIRE day)
# We will use your best ML baseline (the LightGBM one that scored 0.0034) to fill any stubborn gaps.
print("Step 3: Checking for remaining NaNs and filling with best baseline...")
if df_iv_filled.isna().sum().sum() > 0:
    print("Some entire columns are missing. Patching with LightGBM baseline...")
    # Load the best scoring ML matrix we had
    df_lgbm = pd.read_csv('lgbm_boosted_ensemble.csv')
    df_lgbm['datetime'] = pd.to_datetime(df_lgbm['datetime'], format='%d-%m-%Y %H:%M')
    df_lgbm = df_lgbm.set_index('datetime').drop(columns=['underlying_price'])
    
    # Fill the remaining holes
    df_iv_filled = df_iv_filled.fillna(df_lgbm)

# Reconstruct the matrix
df_iv_filled.insert(0, 'underlying_price', underlying)
df_final = df_iv_filled.reset_index()

# Save to disk
df_final.to_csv('time_series_interpolation.csv', index=False)
print("✅ Success! Perfect Time-Series Matrix saved to 'time_series_interpolation.csv'")

Step 1: Loading Original Dataset...
Step 2: Applying Strict Time-Series Interpolation...
Step 3: Checking for remaining NaNs and filling with best baseline...
✅ Success! Perfect Time-Series Matrix saved to 'time_series_interpolation.csv'


In [24]:
import pandas as pd

ORIGINAL_DATASET_PATH = "dataset.csv" 
FILLED_DATASET_PATH = "time_series_interpolation.csv"
OUTPUT_SUBMISSION_PATH = "timeseries_final_submission.csv" 
SEPARATOR = "||"

def generate_solution(filled_path: str, output_path: str):
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)
    feature_cols = [c for c in original.columns if c != "datetime"]
    rows = []
    
    for col in feature_cols:
        was_missing = original[col].isna()
        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)
    solution.to_csv(output_path, index=False)
    print(f"\n✅ Solution saved → {output_path}  ({len(solution)} rows)")

generate_solution(FILLED_DATASET_PATH, OUTPUT_SUBMISSION_PATH)


✅ Solution saved → timeseries_final_submission.csv  (5460 rows)


In [25]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("Step 1: Establishing Time-Series Baseline...")
df_orig = pd.read_csv('dataset.csv')
df_orig['datetime'] = pd.to_datetime(df_orig['datetime'], format='%d-%m-%Y %H:%M')
df_orig = df_orig.sort_values('datetime')

# Create the Time-Series Interpolated matrix
df_time = df_orig.set_index('datetime').copy()
underlying = df_time['underlying_price']
df_iv = df_time.drop(columns=['underlying_price'])

df_iv_time = df_iv.interpolate(method='time', limit_direction='both')
df_iv_time = df_iv_time.bfill().ffill() # Catch any absolute edges

print("Step 2: Reshaping for Residual Analysis...")
df_time_reset = df_iv_time.reset_index()
df_time_reset.insert(1, 'underlying_price', underlying.values)

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df_orig.columns if col not in id_vars]

m_orig = pd.melt(df_orig, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='True_IV')
m_time = pd.melt(df_time_reset, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='Base_IV')

df_meta = pd.merge(m_orig, m_time, on=['datetime', 'underlying_price', 'Ticker'])

print("Step 3: Extracting Market Dynamics Features...")
df_meta['Strike'] = df_meta['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df_meta['Moneyness'] = df_meta['Strike'] / df_meta['underlying_price']
df_meta['Is_Call'] = (df_meta['Ticker'].str[-2:] == 'CE').astype(int)

expiry_str = df_meta['Ticker'].str.extract(r'NIFTY(\d{2}[A-Z]{3}\d{2})')[0]
df_meta['Expiry_Date'] = pd.to_datetime(expiry_str, format='%d%b%y')
df_meta['TTM'] = (df_meta['Expiry_Date'] - df_meta['datetime']).dt.total_seconds() / (365.25 * 24 * 3600)
df_meta['TTM'] = df_meta['TTM'].clip(lower=1/365.25)

# Calculate the precise error of the Time-Series baseline
known_mask = df_meta['True_IV'].notna()
missing_mask = df_meta['True_IV'].isna()

df_meta['Residual'] = np.nan
df_meta.loc[known_mask, 'Residual'] = df_meta.loc[known_mask, 'True_IV'] - df_meta.loc[known_mask, 'Base_IV']

print("Step 4: Training LightGBM to correct Time-Series flaws...")
# Features include the underlying price so the model knows when the market jumped!
features = ['Base_IV', 'underlying_price', 'Moneyness', 'TTM', 'Is_Call']

X_train = df_meta.loc[known_mask, features]
y_train = df_meta.loc[known_mask, 'Residual']
X_predict = df_meta.loc[missing_mask, features]

# Hyper-parameters built specifically for micro-corrections
model = lgb.LGBMRegressor(
    n_estimators=1200, 
    learning_rate=0.01, 
    max_depth=7, 
    num_leaves=63, 
    random_state=42, 
    n_jobs=-1,
    verbose=-1
)
model.fit(X_train, y_train)

print("Step 5: Applying Micro-Corrections and Reconstructing Matrix...")
predicted_residuals = model.predict(X_predict)

df_meta['Final_Predicted_IV'] = df_meta['True_IV']
df_meta.loc[missing_mask, 'Final_Predicted_IV'] = df_meta.loc[missing_mask, 'Base_IV'] + predicted_residuals
df_meta['Final_Predicted_IV'] = df_meta['Final_Predicted_IV'].clip(lower=0.001, upper=3.0)

# Convert datetime back to string format for Kaggle
df_meta['datetime'] = df_meta['datetime'].dt.strftime('%d-%m-%Y %H:%M')

df_final_wide = df_meta.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='Final_Predicted_IV').reset_index()
df_final_wide = df_final_wide[df_orig.columns]

df_final_wide.to_csv('hybrid_ts_ml_ensemble.csv', index=False)
print("✅ Success! Hybrid Matrix saved to 'hybrid_ts_ml_ensemble.csv'")

Step 1: Establishing Time-Series Baseline...
Step 2: Reshaping for Residual Analysis...
Step 3: Extracting Market Dynamics Features...
Step 4: Training LightGBM to correct Time-Series flaws...
Step 5: Applying Micro-Corrections and Reconstructing Matrix...
✅ Success! Hybrid Matrix saved to 'hybrid_ts_ml_ensemble.csv'


In [26]:
import pandas as pd

ORIGINAL_DATASET_PATH = "dataset.csv" 
FILLED_DATASET_PATH = "hybrid_ts_ml_ensemble.csv"
OUTPUT_SUBMISSION_PATH = "hybrid_final_submission.csv" 
SEPARATOR = "||"

def generate_solution(filled_path: str, output_path: str):
    original = pd.read_csv(ORIGINAL_DATASET_PATH)
    filled   = pd.read_csv(filled_path)
    feature_cols = [c for c in original.columns if c != "datetime"]
    rows = []
    
    for col in feature_cols:
        was_missing = original[col].isna()
        for idx in original.index[was_missing]:
            dt  = original.loc[idx, "datetime"]
            uid = f"{dt}{SEPARATOR}{col}"
            val = filled.loc[idx, col]
            rows.append({"id": uid, "value": val})

    solution = pd.DataFrame(rows, columns=["id", "value"])
    solution = solution.sort_values("id").reset_index(drop=True)
    solution.to_csv(output_path, index=False)
    print(f"\n✅ Solution saved → {output_path}  ({len(solution)} rows)")

generate_solution(FILLED_DATASET_PATH, OUTPUT_SUBMISSION_PATH)


✅ Solution saved → hybrid_final_submission.csv  (5460 rows)


In [27]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("--- INITIATING 3D VOLATILITY SURFACE RECONSTRUCTION ---")
df_orig = pd.read_csv('dataset.csv')
df = df_orig.copy()

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df.columns if col not in id_vars]

# ==========================================
# AXIS 1: Z-AXIS (PUT-CALL PARITY)
# ==========================================
print("Step 1: Enforcing Put-Call Parity (Zero-Error Fills)...")
base_tickers = set([col[:-2] for col in ticker_cols])
for base in base_tickers:
    ce_col = base + 'CE'
    pe_col = base + 'PE'
    if ce_col in df.columns and pe_col in df.columns:
        # A missing Call takes the Put's IV, and vice versa
        df[ce_col] = df[ce_col].fillna(df[pe_col])
        df[pe_col] = df[pe_col].fillna(df[ce_col])

# ==========================================
# AXIS 2: X-AXIS (SMILE INTERPOLATION)
# ==========================================
print("Step 2: Interpolating Volatility Smile (Horizontal/Cross-Sectional)...")
# Melt to long format to access the Strike axis mathematically
df_long = pd.melt(df, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='IV')
df_long['Strike'] = df_long['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df_long['Type'] = df_long['Ticker'].str[-2:]
df_long['Expiry'] = df_long['Ticker'].str.extract(r'NIFTY(\d{2}[A-Z]{3}\d{2})')[0]

# Sort to perfectly align the strikes
df_long = df_long.sort_values(['datetime', 'Expiry', 'Type', 'Strike'])
df_long = df_long.set_index('Strike')

# Interpolate strictly *between* known strikes at that exact millisecond
def safe_smile_interpolate(x):
    if len(x.dropna()) > 1:
        return x.interpolate(method='index', limit_area='inside')
    return x

df_long['IV'] = df_long.groupby(['datetime', 'Expiry', 'Type'])['IV'].transform(safe_smile_interpolate)

# Reconstruct back to wide matrix
df_long = df_long.reset_index()
df_wide = df_long.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='IV').reset_index()
df_wide = df_wide[df_orig.columns]

# ==========================================
# AXIS 3: Y-AXIS (TIME INTERPOLATION)
# ==========================================
print("Step 3: Interpolating Term Structure (Vertical/Time-Series)...")
df_wide['datetime'] = pd.to_datetime(df_wide['datetime'], format='%d-%m-%Y %H:%M')
df_wide = df_wide.sort_values('datetime').set_index('datetime')
underlying = df_wide['underlying_price']
df_iv = df_wide.drop(columns=['underlying_price'])

# Connect the remaining gaps over time (This is the logic that secured your 0.0020 score)
df_iv_filled = df_iv.interpolate(method='time', limit_direction='both')
df_iv_filled = df_iv_filled.bfill().ffill() # Absolute edge-case catch

# Final matrix cleanup
df_final = df_iv_filled.reset_index()
df_final.insert(1, 'underlying_price', underlying.values)
df_final['datetime'] = df_final['datetime'].dt.strftime('%d-%m-%Y %H:%M')

# ==========================================
# FINAL STEP: KAGGLE FORMATTER
# ==========================================
print("Step 4: Generating Kaggle Submission Format...")
SEPARATOR = "||"
rows = []

# High-speed exact lookup index
df_final_lookup = df_final.set_index('datetime')

for col in ticker_cols:
    was_missing = df_orig[col].isna()
    for idx in df_orig.index[was_missing]:
        dt  = df_orig.loc[idx, "datetime"]
        uid = f"{dt}{SEPARATOR}{col}"
        val = df_final_lookup.loc[dt, col]
        rows.append({"id": uid, "value": val})

solution = pd.DataFrame(rows, columns=["id", "value"])
solution = solution.sort_values("id").reset_index(drop=True)

# Save the ultimate file
submission_name = 'ULTIMATE_PERFECT_SUBMISSION.csv'
solution.to_csv(submission_name, index=False)
print(f"\n✅ SUCCESS! Master Solution saved to '{submission_name}' ({len(solution)} rows)")

--- INITIATING 3D VOLATILITY SURFACE RECONSTRUCTION ---
Step 1: Enforcing Put-Call Parity (Zero-Error Fills)...
Step 2: Interpolating Volatility Smile (Horizontal/Cross-Sectional)...
Step 3: Interpolating Term Structure (Vertical/Time-Series)...
Step 4: Generating Kaggle Submission Format...

✅ SUCCESS! Master Solution saved to 'ULTIMATE_PERFECT_SUBMISSION.csv' (5460 rows)


In [28]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("--- INITIATING ULTIMATE 3D VOLATILITY SURFACE RECONSTRUCTION ---")

# ==========================================
# STEP 0: LOAD DATA
# ==========================================
print("Step 0: Loading original dataset...")
df_orig = pd.read_csv('dataset.csv')
df = df_orig.copy()

id_vars = ['datetime', 'underlying_price']
ticker_cols = [col for col in df.columns if col not in id_vars]

# ==========================================
# AXIS 1: Z-AXIS (PUT-CALL PARITY)
# ==========================================
print("Step 1: Enforcing Put-Call Parity (Zero-Error Fills)...")
base_tickers = set([col[:-2] for col in ticker_cols])
for base in base_tickers:
    ce_col = base + 'CE'
    pe_col = base + 'PE'
    if ce_col in df.columns and pe_col in df.columns:
        # A missing Call takes the Put's IV, and vice versa perfectly
        df[ce_col] = df[ce_col].fillna(df[pe_col])
        df[pe_col] = df[pe_col].fillna(df[ce_col])

# ==========================================
# AXIS 2: X-AXIS (CUBIC SPLINE SMILE INTERPOLATION)
# ==========================================
print("Step 2: PCHIP Spline Interpolation of Volatility Smile (Cross-Sectional)...")
# Melt to long format to access the Strike axis mathematically
df_long = pd.melt(df, id_vars=id_vars, value_vars=ticker_cols, var_name='Ticker', value_name='IV')
df_long['Strike'] = df_long['Ticker'].str.extract(r'(\d{5})(?:CE|PE)$').astype(float)
df_long['Type'] = df_long['Ticker'].str[-2:]
df_long['Expiry'] = df_long['Ticker'].str.extract(r'NIFTY(\d{2}[A-Z]{3}\d{2})')[0]

# Sort to perfectly align the strikes horizontally
df_long = df_long.sort_values(['datetime', 'Expiry', 'Type', 'Strike'])
df_long = df_long.set_index('Strike')

# Interpolate strictly *between* known strikes using a Parabolic/Cubic Curve
def spline_smile_interpolate(x):
    if len(x.dropna()) > 2:
        # PCHIP maps the parabolic curve without wildly overshooting
        return x.interpolate(method='pchip', limit_area='inside')
    elif len(x.dropna()) > 1:
        # Fallback to linear if only 2 points exist
        return x.interpolate(method='index', limit_area='inside')
    return x

# Apply the curve across the strikes
df_long['IV'] = df_long.groupby(['datetime', 'Expiry', 'Type'])['IV'].transform(spline_smile_interpolate)

# Reconstruct back to wide matrix
df_long = df_long.reset_index()
df_wide = df_long.pivot(index=['datetime', 'underlying_price'], columns='Ticker', values='IV').reset_index()
df_wide = df_wide[df_orig.columns]

# ==========================================
# AXIS 3: Y-AXIS (TIME INTERPOLATION)
# ==========================================
print("Step 3: Interpolating Term Structure (Vertical/Time-Series)...")
# Convert to proper datetime objects to measure exact time gaps
df_wide['datetime'] = pd.to_datetime(df_wide['datetime'], format='%d-%m-%Y %H:%M')
df_wide = df_wide.sort_values('datetime').set_index('datetime')

underlying = df_wide['underlying_price']
df_iv = df_wide.drop(columns=['underlying_price'])

# Connect the remaining gaps over time (The logic that originally secured 0.0020)
df_iv_filled = df_iv.interpolate(method='time', limit_direction='both')
df_iv_filled = df_iv_filled.bfill().ffill() # Absolute edge-case catch

# Final matrix reconstruction
df_final = df_iv_filled.reset_index()
df_final.insert(1, 'underlying_price', underlying.values)
df_final['datetime'] = df_final['datetime'].dt.strftime('%d-%m-%Y %H:%M')

# ==========================================
# FINAL STEP: KAGGLE FORMATTER
# ==========================================
print("Step 4: Generating Exact Kaggle Submission Format...")
SEPARATOR = "||"
rows = []

# High-speed exact lookup index
df_final_lookup = df_final.set_index('datetime')

# Loop strictly through the values that were originally missing
for col in ticker_cols:
    was_missing = df_orig[col].isna()
    for idx in df_orig.index[was_missing]:
        # Get the exact original datetime string to ensure Kaggle's IDs match perfectly
        dt  = df_orig.loc[idx, "datetime"]
        uid = f"{dt}{SEPARATOR}{col}"
        val = df_final_lookup.loc[dt, col]
        rows.append({"id": uid, "value": val})

# Build the final clean dataframe
solution = pd.DataFrame(rows, columns=["id", "value"])
solution = solution.sort_values("id").reset_index(drop=True)

# Save the ultimate file
submission_name = 'ULTIMATE_SPLINE_SUBMISSION.csv'
solution.to_csv(submission_name, index=False)
print(f"\n✅ SUCCESS! Master Solution saved to '{submission_name}' ({len(solution)} rows)")

--- INITIATING ULTIMATE 3D VOLATILITY SURFACE RECONSTRUCTION ---
Step 0: Loading original dataset...
Step 1: Enforcing Put-Call Parity (Zero-Error Fills)...
Step 2: PCHIP Spline Interpolation of Volatility Smile (Cross-Sectional)...
Step 3: Interpolating Term Structure (Vertical/Time-Series)...
Step 4: Generating Exact Kaggle Submission Format...

✅ SUCCESS! Master Solution saved to 'ULTIMATE_SPLINE_SUBMISSION.csv' (5460 rows)


In [1]:
from acnportal.acndata import DataClient

API_KEY = "cALBXXl0_PcnWBrSQBeMJm_8ceG7PxemKrUXIuyJBCA"

client = DataClient(API_KEY)

In [3]:
from acnportal.acndata import DataClient

client = DataClient("cALBXXl0_PcnWBrSQBeMJm_8ceG7PxemKrUXIuyJBCA")

sessions = client.get_sessions(site="caltech")

first_session = next(sessions)

print(first_session)

{'_id': '5bc90cb9f9af8b0d7fe77cd2', 'userInputs': None, 'sessionID': '2_39_78_362_2018-04-25 11:08:04.400812', 'stationID': '2-39-78-362', 'spaceID': 'CA-496', 'siteID': '0002', 'clusterID': '0039', 'connectionTime': datetime.datetime(2018, 4, 25, 4, 8, 4, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'disconnectTime': datetime.datetime(2018, 4, 25, 6, 20, 10, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'kWhDelivered': 7.932, 'doneChargingTime': datetime.datetime(2018, 4, 25, 6, 21, 10, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'timezone': 'America/Los_Angeles', 'userID': None}


In [4]:
import pandas as pd
from acnportal.acndata import DataClient

client = DataClient("cALBXXl0_PcnWBrSQBeMJm_8ceG7PxemKrUXIuyJBCA")

records = []

for session in client.get_sessions(site="caltech"):
    records.append(session)

df = pd.DataFrame(records)

print(df.head())
print(df.shape)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [5]:
from acnportal.acndata import DataClient

client = DataClient("cALBXXl0_PcnWBrSQBeMJm_8ceG7PxemKrUXIuyJBCA")

sessions = client.get_sessions(site="caltech")

print(type(sessions))

<class 'generator'>


In [7]:
from acnportal.acndata import DataClient

client = DataClient("cALBXXl0_PcnWBrSQBeMJm_8ceG7PxemKrUXIuyJBCA")

sessions = client.get_sessions(site="caltech")

first_session = next(sessions)

print(first_session)

{'_id': '5bc90cb9f9af8b0d7fe77cd2', 'userInputs': None, 'sessionID': '2_39_78_362_2018-04-25 11:08:04.400812', 'stationID': '2-39-78-362', 'spaceID': 'CA-496', 'siteID': '0002', 'clusterID': '0039', 'connectionTime': datetime.datetime(2018, 4, 25, 4, 8, 4, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'disconnectTime': datetime.datetime(2018, 4, 25, 6, 20, 10, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'kWhDelivered': 7.932, 'doneChargingTime': datetime.datetime(2018, 4, 25, 6, 21, 10, tzinfo=<DstTzInfo 'America/Los_Angeles' PDT-1 day, 17:00:00 DST>), 'timezone': 'America/Los_Angeles', 'userID': None}


In [8]:
import acnportal

print(acnportal.__version__)

AttributeError: module 'acnportal' has no attribute '__version__'

In [10]:
first_session = next(client.get_sessions(site="caltech"))

In [12]:
import pandas as pd
from acnportal.acndata import DataClient

client = DataClient("cALBXXl0_PcnWBrSQBeMJm_8ceG7PxemKrUXIuyJBCA")

records = []

for i, session in enumerate(client.get_sessions(site="caltech")):
    records.append(session)

    if i == 999:
        break

df = pd.DataFrame(records)

print(df.shape)
df.head()

(1000, 13)


,_id,userInputs,sessionID,stationID,spaceID,siteID,clusterID,connectionTime,disconnectTime,kWhDelivered,doneChargingTime,timezone,userID
0,5bc90cb9f9af8b0d7fe77cd2,None,2_39_78_362_2018-04-25 11:08:04.400812,2-39-78-362,CA-496,0002,0039,2018-04-25 04:08:04-07:00,2018-04-25 06:20:10-07:00,7.932,2018-04-25 06:21:10-07:00,America/Los_Angeles,NaN
1,5bc90cb9f9af8b0d7fe77cd3,None,2_39_95_27_2018-04-25 13:45:09.617470,2-39-95-27,CA-319,0002,0039,2018-04-25 06:45:10-07:00,2018-04-25 17:56:16-07:00,10.013,2018-04-25 09:44:15-07:00,America/Los_Angeles,NaN
2,5bc90cb9f9af8b0d7fe77cd4,None,2_39_79_380_2018-04-25 13:45:49.962001,2-39-79-380,CA-489,0002,0039,2018-04-25 06:45:50-07:00,2018-04-25 16:04:45-07:00,5.257,2018-04-25 07:51:44-07:00,America/Los_Angeles,NaN
3,5bc90cb9f9af8b0d7fe77cd5,None,2_39_79_379_2018-04-25 14:37:06.460772,2-39-79-379,CA-327,0002,0039,2018-04-25 07:37:06-07:00,2018-04-25 16:55:34-07:00,5.177,2018-04-25 09:05:22-07:00,America/Los_Angeles,NaN
4,5bc90cb9f9af8b0d7fe77cd6,None,2_39_79_381_2018-04-25 14:40:33.638896,2-39-79-381,CA-490,0002,0039,2018-04-25 07:40:34-07:00,2018-04-25 16:03:12-07:00,10.119,2018-04-25 10:40:30-07:00,America/Los_Angeles,NaN


In [13]:
df.to_csv("caltech_1000_sessions.csv", index=False)

print("Saved!")

Saved!


In [14]:
print(df.columns.tolist())

['_id', 'userInputs', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'connectionTime', 'disconnectTime', 'kWhDelivered', 'doneChargingTime', 'timezone', 'userID']


In [15]:
df.isnull().sum()

_id                   0
userInputs          996
sessionID             0
stationID             0
spaceID               0
siteID                0
clusterID             0
connectionTime        0
disconnectTime        0
kWhDelivered          0
doneChargingTime      1
timezone              0
userID              996
dtype: int64

In [16]:
df.describe(include="all")

,_id,userInputs,sessionID,stationID,spaceID,siteID,clusterID,connectionTime,disconnectTime,kWhDelivered,doneChargingTime,timezone,userID
count,1000,4,1000,1000,1000,1000,1000,1000,1000,1000.000000,999,1000,4
unique,1000,4,1000,51,51,1,1,NaN,NaN,NaN,NaN,1,3
top,5bc90cb9f9af8b0d7fe77cd2,"[{'userID': 22, 'milesRequested': 170, 'WhPerM...",2_39_78_362_2018-04-25 11:08:04.400812,2-39-139-28,CA-303,0002,0039,NaN,NaN,NaN,NaN,America/Los_Angeles,000000022
freq,1,1,1,47,47,1000,1000,NaN,NaN,NaN,NaN,1000,2
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-05-05 00:05:04.169000-07:00,2018-05-05 06:29:15.885999-07:00,8.997422,2018-05-05 03:39:27.125125-07:00,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-04-25 04:08:04-07:00,2018-04-25 06:20:10-07:00,0.535000,2018-04-25 06:21:10-07:00,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-04-30 13:20:45.750000-07:00,2018-04-30 18:28:32.750000-07:00,3.822250,2018-04-30 17:04:29.500000-07:00,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-05-04 16:40:29.500000-07:00,2018-05-04 21:02:40.500000-07:00,7.096500,2018-05-04 19:08:54-07:00,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-05-09 16:28:52.500000-07:00,2018-05-09 19:55:10.500000-07:00,12.851000,2018-05-09 18:42:55.500000-07:00,NaN,NaN
max,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-05-14 14:52:49-07:00,2018-05-14 20:08:15-07:00,47.808000,2018-05-14 17:57:33-07:00,NaN,NaN


In [18]:
import pandas as pd
from acnportal.acndata import DataClient

client = DataClient("cALBXXl0_PcnWBrSQBeMJm_8ceG7PxemKrUXIuyJBCA")

records = []

for i, session in enumerate(client.get_sessions(site="caltech")):

    records.append(session)

    if (i + 1) % 5000 == 0:
        pd.DataFrame(records).to_csv(
            f"caltech_{i+1}_sessions.csv",
            index=False
        )
        print(f"Saved {i+1} sessions")

    if i == 49999:
        break

df = pd.DataFrame(records)
df.to_csv("caltech_50000_sessions_final.csv", index=False)

print("Done!")

Saved 5000 sessions


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [20]:
import pandas as pd
from acnportal.acndata import DataClient

client = DataClient("cALBXXl0_PcnWBrSQBeMJm_8ceG7PxemKrUXIuyJBCA")

chunk_size = 5000
chunk = []
file_num = 1

try:
    for i, session in enumerate(client.get_sessions(site="caltech")):

        chunk.append(session)

        if len(chunk) >= chunk_size:
            df = pd.DataFrame(chunk)

            filename = f"caltech_chunk_{file_num}.csv"
            df.to_csv(filename, index=False)

            print(f"Saved {filename}")

            chunk = []
            file_num += 1

except Exception as e:
    print("Stopped due to error:")
    print(e)

    if len(chunk) > 0:
        df = pd.DataFrame(chunk)
        filename = f"caltech_chunk_{file_num}_partial.csv"
        df.to_csv(filename, index=False)
        print(f"Saved partial chunk: {filename}")

Saved caltech_chunk_1.csv
Saved caltech_chunk_2.csv
Saved caltech_chunk_3.csv
Stopped due to error:
Expecting value: line 1 column 1 (char 0)
Saved partial chunk: caltech_chunk_4_partial.csv


In [21]:
import pandas as pd

df1 = pd.read_csv("caltech_chunk_1.csv")
df2 = pd.read_csv("caltech_chunk_2.csv")
df3 = pd.read_csv("caltech_chunk_3.csv")

merged = pd.concat([df1, df2, df3], ignore_index=True)

merged = merged.drop_duplicates()

print(merged.shape)

merged.to_csv("caltech.csv", index=False)

print("Saved as caltech.csv")

(15000, 13)
Saved as caltech.csv
